
# Milvus on Zilliz Cloud

This notebook shows **how to connect directly to a cloud‑hosted Milvus database (Zilliz Cloud)** and perform:

- Connection & authentication
- Collection inspection
- CRUD operations
- Vector similarity search
- Visual exploration of embeddings (2D)

Designed for **teaching Milvus concepts**, not local Docker setups.



## 1 Install & Import Dependencies


In [1]:
%pip install pymilvus

Note: you may need to restart the kernel to use updated packages.


In [2]:

%pip install numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



## 2 Connect to Zilliz Cloud (Milvus)



In [3]:
import os
from dotenv import load_dotenv
# from pymilvus import connections

# # If using Docker standalone Milvus
# connections.connect("default", host="127.0.0.1", port="19530")

from pymilvus import connections

load_dotenv(override=True, dotenv_path="../.env.local")

milvus_uri = os.getenv("MILVUS_URI")
milvus_token = os.getenv("MILVUS_API_KEY")


connections.connect(
    alias="default",
    uri=milvus_uri,
    token=milvus_token
)

print("Connected to Milvus on Zilliz Cloud")


Connected to Milvus on Zilliz Cloud



## 3 Inspect Collections


In [4]:

from pymilvus import utility

utility.list_collections()


['demo_collection',
 'policy_docs_collection',
 'collection_1',
 'test_collection']


## 4 Load & Inspect a Collection


In [5]:

from pymilvus import Collection

# Collection is same as a Table in traditional databases
collection = Collection("demo_collection")
collection.load()

collection.schema


{'auto_id': True, 'description': 'demo_collection', 'fields': [{'name': 'id', 'description': 'The Primary Key', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 4}}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}], 'enable_dynamic_field': True, 'enable_namespace': False}


## 5 Read Data (Query)

Query a few rows to see **raw stored data**


In [6]:

results = collection.query(
    expr="id >= 0",
    
    output_fields=["id", "title", "vector"],
    limit=5
)
# select id, title, vector from demo_collection where id >= 0 limit 5
results


data: ["{'id': 463705163763347398, 'title': 'Intro to AI', 'vector': [0.7567089796066284, 0.056757595390081406, 0.8686589002609253, 0.5566412210464478]}", "{'id': 463705163763347400, 'title': 'Vector Databases', 'vector': [0.34350088238716125, 0.34113776683807373, 0.11458902806043625, 0.6653984189033508]}", "{'id': 463705164234735154, 'title': 'Deep Learning (Updated)', 'vector': [0.7000895142555237, 0.022113775834441185, 0.48144587874412537, 0.23203983902931213]}", "{'id': 463705164234754808, 'title': 'Deep Learning (Updated)', 'vector': [0.7000895142555237, 0.022113775834441185, 0.48144587874412537, 0.23203983902931213]}", "{'id': 463705164369185718, 'title': 'Milvus makes vector search scalable', 'vector': [0.9826512932777405, 0.9719760417938232, 0.9610936641693115, 0.3447304368019104]}"], extra_info: {'cost': 6, 'scanned_remote_bytes': 2097152, 'scanned_total_bytes': 3145728, 'cache_hit_ratio': 0.3333333333333333}

In [7]:
results = collection.query(
    expr="id in [463705163763347400, 463705164234735154] AND title == 'Deep Learning (Updated)'",
    
    output_fields=["id", "title", "vector"],
    limit=5
)
# select id, title, vector from demo_collection where id >= 0 limit 5
results


data: ["{'id': 463705164234735154, 'title': 'Deep Learning (Updated)', 'vector': [0.7000895142555237, 0.022113775834441185, 0.48144587874412537, 0.23203983902931213]}"], extra_info: {'cost': 6, 'scanned_remote_bytes': 0, 'scanned_total_bytes': 5242880, 'cache_hit_ratio': 1.0}


## 6 Insert (CREATE)

Insert a new vector record


In [8]:
import numpy as np

data = [
    [[.3456, .2345, .1234, .5678]],            # vector FIRST
    ["Milvus makes vector search scalable"]     # title SECOND
]

collection.insert(data)
collection.flush()



## 7 Update (DELETE + INSERT pattern)

Milvus does not support in‑place updates.


In [9]:
# collection.delete(expr="id == 463705163763347399")
# collection.flush()

updated_data = [
    [463705165204270561],  # ← list of IDs (1 row)
    [
        [0.7000895, 0.022113776, 0.48144588, 0.23203984]
    ],  # ← list of vectors (1 row)
    [
        "Next.JS for Beginners - Updated"
    ]   # ← list of titles (1 row)
]

result = collection.upsert(updated_data)
collection.flush()

new_id = result.primary_keys[0]
print(f"Record updated. New ID generated: {new_id}")


Record updated. New ID generated: 464516272861398375



## 8 Delete


In [10]:

collection.delete(expr="id == 463705164234735372")
collection.flush()

print("Record deleted")



Record deleted


In [11]:

results = collection.query(
    expr="id == 463705164234754808",
    output_fields=[ "title", "vector"],
    # output_fields=["*"],
    limit=5
)

results


data: ["{'title': 'Deep Learning (Updated)', 'vector': [0.7000895142555237, 0.022113775834441185, 0.48144587874412537, 0.23203983902931213], 'id': 463705164234754808}"], extra_info: {'cost': 6, 'scanned_remote_bytes': 0, 'scanned_total_bytes': 3145728, 'cache_hit_ratio': 1.0}


## 9 Vector Similarity Search


In [ ]:
query_vector = [0.4670895, 0.343513776, 0.22224588, 0.113984]
print(f"Query Vector: {query_vector}")
search_results = collection.search(
    data=[query_vector],
    anns_field="vector",
    param={"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=2,
    output_fields=["id","title"]
)
# SELECT id, title FROM demo_collection WHERE COSINE_SIMILARITY(embedding, [0.4670895, 0.343513776, 0.22224588, 0.113984]) > threshold LIMIT 2
print(f"Search Results: {search_results}")

for hit in search_results[0]:
    print(f"title={hit.entity.get('title')}")



## 10 Visualizing Embeddings (PCA)

This helps students **see vector similarity**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 1️⃣ Pull vectors + titles from Milvus
data = collection.query(
    expr="id >= 0",
    output_fields=["vector", "title"],
    limit=100
)

vectors = np.array([d["vector"] for d in data])
titles = [d["title"] for d in data]

# 2️⃣ Reduce vectors to 2D using PCA
pca = PCA(n_components=2)
reduced = pca.fit_transform(vectors)

# 3️⃣ Color points by title
unique_titles = list(set(titles))
colors = plt.cm.tab10(range(len(unique_titles)))
color_map = dict(zip(unique_titles, colors))

plt.figure(figsize=(8,6))

for i, title in enumerate(titles):
    plt.scatter(
        reduced[i, 0],
        reduced[i, 1],
        color=color_map[title],
        alpha=0.7,
        label=title
    )

# Remove duplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), fontsize=8)

plt.title("Vector Visualization (PCA, Colored by Title)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()
